In [39]:
from decimal import Decimal
import pandas as pd
from calendar import month_abbr
from luca.taxes import TaxSchedule

In [23]:
one_cent = Decimal('0.01')
def c(value):
    """Convert 'value' to Decimal cents."""
    return Decimal(value).quantize(one_cent)

In [24]:
c(14.95)

Decimal('14.95')

In [25]:
df = pd.DataFrame(index=month_abbr[1:])
df['wage'] = c('11000.00')
print(df)

         wage
Jan  11000.00
Feb  11000.00
Mar  11000.00
Apr  11000.00
May  11000.00
Jun  11000.00
Jul  11000.00
Aug  11000.00
Sep  11000.00
Oct  11000.00
Nov  11000.00
Dec  11000.00


In [26]:
federal_monthly = TaxSchedule([
        (325, '10'), (1023, '15'), (3163, '25'), (6050, '28'),
        (9050, '33'), (15906, '35'), (17925, '39.6'),
    ])

In [27]:
df['fedwh'] = df['wage'].map(federal_monthly.compute_tax_on)
print(df)

         wage    fedwh
Jan  11000.00  2596.05
Feb  11000.00  2596.05
Mar  11000.00  2596.05
Apr  11000.00  2596.05
May  11000.00  2596.05
Jun  11000.00  2596.05
Jul  11000.00  2596.05
Aug  11000.00  2596.05
Sep  11000.00  2596.05
Oct  11000.00  2596.05
Nov  11000.00  2596.05
Dec  11000.00  2596.05


In [28]:
ss_limit = c('117000.00')
ss_rate = Decimal('0.062')
mc_rate = Decimal('0.0145')

print(df['wage'].cumsum())

Jan     11000.00
Feb     22000.00
Mar     33000.00
Apr     44000.00
May     55000.00
Jun     66000.00
Jul     77000.00
Aug     88000.00
Sep     99000.00
Oct    110000.00
Nov    121000.00
Dec    132000.00
Name: wage, dtype: object


In [29]:
cum_ss_wage = df['wage'].cumsum().clip(upper=ss_limit)
print(cum_ss_wage)

Jan     11000.00
Feb     22000.00
Mar     33000.00
Apr     44000.00
May     55000.00
Jun     66000.00
Jul     77000.00
Aug     88000.00
Sep     99000.00
Oct    110000.00
Nov    117000.00
Dec    117000.00
Name: wage, dtype: object


In [30]:
ss_wage = cum_ss_wage.diff().fillna(df['wage'])
print(ss_wage)

Jan    11000.00
Feb    11000.00
Mar    11000.00
Apr    11000.00
May    11000.00
Jun    11000.00
Jul    11000.00
Aug    11000.00
Sep    11000.00
Oct    11000.00
Nov     7000.00
Dec        0.00
Name: wage, dtype: object


In [31]:
df['ss_tax'] = (ss_wage * ss_rate).map(c)
df['mc_tax'] = (df['wage'] * mc_rate).map(c)
print(df)

         wage    fedwh  ss_tax  mc_tax
Jan  11000.00  2596.05  682.00  159.50
Feb  11000.00  2596.05  682.00  159.50
Mar  11000.00  2596.05  682.00  159.50
Apr  11000.00  2596.05  682.00  159.50
May  11000.00  2596.05  682.00  159.50
Jun  11000.00  2596.05  682.00  159.50
Jul  11000.00  2596.05  682.00  159.50
Aug  11000.00  2596.05  682.00  159.50
Sep  11000.00  2596.05  682.00  159.50
Oct  11000.00  2596.05  682.00  159.50
Nov  11000.00  2596.05  434.00  159.50
Dec  11000.00  2596.05    0.00  159.50


In [32]:
ohio_exemption = c('650.00') / Decimal('12')
ohio_monthly = TaxSchedule(
[(0, '0.5'), (5000, '1'), (10000, '2'), (15000, '2.5'), 
 (20000, '3'), (40000, '3.5'), (80000, '4'), (100000, '5')],
ohio_exemption,
)

df['statewh'] = df['wage'].map(ohio_monthly.compute_tax_on)

df['paycheck'] = (df['wage'] - df['fedwh'] - df['ss_tax'] - df['mc_tax'] - df['statewh'])

print(df)

         wage    fedwh  ss_tax  mc_tax statewh paycheck
Jan  11000.00  2596.05  682.00  159.50   95.00  7467.45
Feb  11000.00  2596.05  682.00  159.50   95.00  7467.45
Mar  11000.00  2596.05  682.00  159.50   95.00  7467.45
Apr  11000.00  2596.05  682.00  159.50   95.00  7467.45
May  11000.00  2596.05  682.00  159.50   95.00  7467.45
Jun  11000.00  2596.05  682.00  159.50   95.00  7467.45
Jul  11000.00  2596.05  682.00  159.50   95.00  7467.45
Aug  11000.00  2596.05  682.00  159.50   95.00  7467.45
Sep  11000.00  2596.05  682.00  159.50   95.00  7467.45
Oct  11000.00  2596.05  682.00  159.50   95.00  7467.45
Nov  11000.00  2596.05  434.00  159.50   95.00  7715.45
Dec  11000.00  2596.05    0.00  159.50   95.00  8149.45


In [33]:
print('September payroll stub:')
print(df.loc['Sep'])

September payroll stub:
wage        11000.00
fedwh        2596.05
ss_tax        682.00
mc_tax        159.50
statewh        95.00
paycheck     7467.45
Name: Sep, dtype: object


In [34]:
print("End of year W-2")
print(df.apply(sum))

End of year W-2
wage        132000.00
fedwh        31152.60
ss_tax        7254.00
mc_tax        1914.00
statewh       1140.00
paycheck     90539.40
dtype: object


In [21]:
df

,wage,fedwh,ss_tax,mc_tax,statewh,paycheck
Jan,11000.00,2596.05,682.00,159.50,95.00,7467.45
Feb,11000.00,2596.05,682.00,159.50,95.00,7467.45
Mar,11000.00,2596.05,682.00,159.50,95.00,7467.45
Apr,11000.00,2596.05,682.00,159.50,95.00,7467.45
May,11000.00,2596.05,682.00,159.50,95.00,7467.45
Jun,11000.00,2596.05,682.00,159.50,95.00,7467.45
Jul,11000.00,2596.05,682.00,159.50,95.00,7467.45
Aug,11000.00,2596.05,682.00,159.50,95.00,7467.45
Sep,11000.00,2596.05,682.00,159.50,95.00,7467.45
Oct,11000.00,2596.05,682.00,159.50,95.00,7467.45


In [35]:
print(df)

         wage    fedwh  ss_tax  mc_tax statewh paycheck
Jan  11000.00  2596.05  682.00  159.50   95.00  7467.45
Feb  11000.00  2596.05  682.00  159.50   95.00  7467.45
Mar  11000.00  2596.05  682.00  159.50   95.00  7467.45
Apr  11000.00  2596.05  682.00  159.50   95.00  7467.45
May  11000.00  2596.05  682.00  159.50   95.00  7467.45
Jun  11000.00  2596.05  682.00  159.50   95.00  7467.45
Jul  11000.00  2596.05  682.00  159.50   95.00  7467.45
Aug  11000.00  2596.05  682.00  159.50   95.00  7467.45
Sep  11000.00  2596.05  682.00  159.50   95.00  7467.45
Oct  11000.00  2596.05  682.00  159.50   95.00  7467.45
Nov  11000.00  2596.05  434.00  159.50   95.00  7715.45
Dec  11000.00  2596.05    0.00  159.50   95.00  8149.45


In [41]:
federal_monthly_withholding = {
    'MJ': TaxSchedule(one_allowance=325, brackets = [
            (692, 10),
            (2179, 15),
            (6733, 25),
            (12892, 28),
            (19279, 33),
            (33888, 35),
            (38192, '39.6'),
            ]),
    }

In [43]:
schedule = federal_monthly_withholding['MJ']
print(schedule.build_instructions())

schedule.compute_tax_on(c(25000))


       over but_not_over       pay   plus of_excess_over
0    692.00      2179.00      0.00   0.10         692.00
1   2179.00      6733.00    148.70   0.15        2179.00
2   6733.00     12892.00    831.80   0.25        6733.00
3  12892.00     19279.00   2371.55   0.28       12892.00
4  19279.00     33888.00   4159.91   0.33       19279.00
5  33888.00     38192.00   8980.88   0.35       33888.00
6  38192.00     Infinity  10487.28  0.396       38192.00


Decimal('6047.84')